In [1]:
# ############################################################
# Level 3 — 파이프라인 + 튜닝 (공개 데이터, 누수 방지)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 표 / 파이프라인 / 전처리 / 모델 / 튜닝
# ------------------------------------------------------------
import pandas as pd                               # 표(CSV) 다루기
from sklearn.pipeline import make_pipeline        # 전처리+모델 한 상자
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

In [2]:
# ------------------------------------------------------------
# [목적] 데이터 불러오기 + 전처리 (글자 -> 0/1)
# ------------------------------------------------------------
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
df = pd.read_csv(url)                              # 공개 URL에서 바로
X = pd.get_dummies(df.drop(columns='charges'))    # 글자 특성 -> 0/1
y = df['charges']                                 # 정답 = 의료비

In [3]:
# ------------------------------------------------------------
# [목적] 파이프라인을 통째로 튜닝 ('스텝이름__파라미터' 형식)
# ------------------------------------------------------------
pipe = make_pipeline(StandardScaler(), Ridge())   # 스케일링+모델 한 상자(누수 방지)
grid = GridSearchCV(pipe, {'ridge__alpha': [0.1, 1, 10, 100]},   # ridge의 alpha 튜닝
                    cv=5, scoring='r2')
grid.fit(X, y)                                    # 파이프라인 전체를 교차검증으로 튜닝
print('최적:', grid.best_params_, '| 교차검증 R2:', round(grid.best_score_, 3))

최적: {'ridge__alpha': 10} | 교차검증 R2: 0.747


In [4]:
# ============================================================
# [결과 해석]
#  · 최적 ridge__alpha=10, 교차검증 R2 ~ 0.747 (의료비 변동의 약 75% 설명)
#  · 파이프라인 덕분에 교차검증의 매 조각마다 스케일링이 안전하게(누수 없이) 적용됨
#  · 'make_pipeline + GridSearchCV' 조합이 실전 표준 (전처리까지 포함해 공정하게 튜닝)
# ============================================================